### Logging

In [1]:
def log(msg):
    # Simple wrapper so you can later swap with logging.info()
    print(f"[INFO] {msg}")

log("Defined logger")

[INFO] Defined logger


### Preprocessing

In [2]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

### CLEANING DATA

class DataCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()

        # Numeric conversion
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')

        # Binary mappings (NO Churn here!)
        binary_map = {
            'gender': {'Female': 1, 'Male': 0},
            'Partner': {'Yes': 1, 'No': 0},
            'Dependents': {'Yes': 1, 'No': 0},
            'PhoneService': {'Yes': 1, 'No': 0},
            'PaperlessBilling': {'Yes': 1, 'No': 0},
        }

        for col, mapping in binary_map.items():
            if col in X:
                X[col] = X[col].map(mapping)

        # MultipleLines
        if 'MultipleLines' in X:
            X['MultipleLines'] = (X['MultipleLines'] == 'Yes').astype(int)

        # Internet-dependent features
        internet_features = [
            'OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies'
        ]

        for col in internet_features:
            if col in X:
                X[col] = (X[col] == 'Yes').astype(int)

        return X
    


### FEATURE ENGINEERING

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_charge_ = X['MonthlyCharges'].median()
        return self

    def transform(self, X):
        X = X.copy()

        # Groups
        X['tenure_group'] = pd.cut(
            X['tenure'],
            bins=[0, 12, 24, 48, 72],
            labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr']
        )

        X['avg_revenue'] = X['TotalCharges'] / (X['tenure'] + 1)

        return X
    
class FlagBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, flag_configs):
        self.flag_configs = flag_configs

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        for new_col, func in self.flag_configs.items():
            X[new_col] = func(X).astype(int)
        return X
    
flag_builder = FlagBuilder({
    'is_new_customer': lambda X: X['tenure'] <= 12,
    'is_long_term': lambda X: X['Contract'].isin(['One year', 'Two year']),
    'auto_pay_flag': lambda X: X['PaymentMethod'].isin([
        'Bank transfer (automatic)', 
        'Credit card (automatic)'
    ]),
    'family_flag': lambda X: (X['Partner'] == 'Yes') | (X['Dependents'] == 'Yes'),
    'fiber_flag': lambda X: X['InternetService'] == 'Fiber optic',
    'electronic_check_flag': lambda X: X['PaymentMethod'] == 'Electronic check',
})


class HighMonthlyFlag(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_ = X['MonthlyCharges'].median()
        return self

    def transform(self, X):
        X = X.copy()
        X['high_monthly_flag'] = (X['MonthlyCharges'] > self.median_).astype(int)
        return X
    

class ServiceCounter(BaseEstimator, TransformerMixin):
    def __init__(self, service_cols):
        self.service_cols = service_cols

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        X['num_services'] = (X[self.service_cols] == 'Yes').sum(axis=1)
        return X
    

class InteractionBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, interactions):
        self.interactions = interactions

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        for new_col, (c1, c2) in self.interactions.items():
            X[new_col] = X[c1] * X[c2]
        return X
    
feature_pipeline = Pipeline([
    ('basic', FeatureEngineer()),
    ('flags', flag_builder),
    ('high_monthly', HighMonthlyFlag()),
    ('services', ServiceCounter([
        'OnlineSecurity','OnlineBackup','DeviceProtection',
        'TechSupport','StreamingTV','StreamingMovies'
    ])),
    ('interactions', InteractionBuilder({
        'new_echeck_interaction': ('is_new_customer', 'electronic_check_flag'),
        'fiber_highcharge_interaction': ('fiber_flag', 'high_monthly_flag'),
        'loyal_engaged_interaction': ('is_long_term', 'num_services'),
    }))
])


## REAUSABLE PREPROCESSING COMPONENTS

median_imputer = SimpleImputer(strategy='median')
most_frequent_imputer = SimpleImputer(strategy='most_frequent')
missing_cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')

scaler = StandardScaler()

ohe_drop_first = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ohe_all = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

## FEATURES

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_revenue']
categorical_features = ['Contract', 'PaymentMethod', 'InternetService', 'tenure_group']
engineered_features = [
    'is_new_customer', 
    'is_long_term', 
    'auto_pay_flag', 
    'num_services',
    'high_monthly_flag', 
    'family_flag', 
    'fiber_flag', 
    'electronic_check_flag',
    'new_echeck_interaction', 
    'fiber_highcharge_interaction', 
    'loyal_engaged_interaction'
]


### LINEAR AND TREE PREPROCESSING

# -------------------------------
# 1. Linear model preprocessing
# -------------------------------
num_pipeline_linear = Pipeline([
    ('imputer', median_imputer),
    ('scaler', scaler)
])

cat_pipeline_linear = Pipeline([
    ('imputer', missing_cat_imputer),
    ('encoder', ohe_drop_first)
])

linear_preprocessor = ColumnTransformer([
    ('num', num_pipeline_linear, numeric_features + engineered_features),
    ('cat', cat_pipeline_linear, categorical_features)
])

# -------------------------------
# 2. Tree-based preprocessing
# -------------------------------
num_pipeline_tree = Pipeline([
    ('imputer', median_imputer)
])

cat_pipeline_tree = Pipeline([
    ('imputer', most_frequent_imputer),
    ('encoder', ohe_all)
])

tree_preprocessor = ColumnTransformer([
    ('num', num_pipeline_tree, numeric_features + engineered_features),
    ('cat', cat_pipeline_tree, categorical_features)
])



## FINAL PIPELINES


full_pipeline_linear = Pipeline([
    ('cleaning', DataCleaner()),
    ('feature_engineering', feature_pipeline),
    ('preprocessing', linear_preprocessor)
])

full_pipeline_tree = Pipeline([
    ('cleaning', DataCleaner()),
    ('feature_engineering', feature_pipeline),
    ('preprocessing', tree_preprocessor)
])

### Training Model

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

data_path = "../data/raw/telco_churn.csv"
df_raw = pd.read_csv(data_path)

X_raw = df_raw.drop(columns='Churn')
y = df_raw['Churn'].map({'Yes': 1, 'No': 0})

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

full_pipeline_linear.fit(X_train_raw, y_train)
X_train_linear = full_pipeline_linear.transform(X_train_raw)
X_test_linear  = full_pipeline_linear.transform(X_test_raw)

full_pipeline_tree.fit(X_train_raw, y_train)
X_train_tree = full_pipeline_tree.transform(X_train_raw)
X_test_tree  = full_pipeline_tree.transform(X_test_raw)





# Instantiate baseline logistic regression
logreg = LogisticRegression(random_state=42, max_iter=1000)

# Fit on preprocessed linear training data
logreg.fit(X_train_linear, y_train)

# Predict
y_pred_logreg = logreg.predict(X_test_linear)
y_proba_logreg = logreg.predict_proba(X_test_linear)[:, 1]


# Instantiate baseline random forest
rf = RandomForestClassifier(random_state=42, n_estimators=100)

# Fit on tree-preprocessed data
rf.fit(X_train_tree, y_train)

# Predict
y_pred_rf = rf.predict(X_test_tree)
y_proba_rf = rf.predict_proba(X_test_tree)[:, 1]

# Instantiate baseline gradient boosting
gb = GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1)

# Fit on tree-preprocessed data
gb.fit(X_train_tree, y_train)

# Predict
y_pred_gb = gb.predict(X_test_tree)
y_proba_gb = gb.predict_proba(X_test_tree)[:, 1]

### Evaluation

In [4]:

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

# Evaluate Logistic Regression
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_logreg))

# Evaluate Random Forest
print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

# Evaluate Gradient Boosting
print("Gradient Boosting")
print("Accuracy:", accuracy_score(y_test, y_pred_gb))

# Model Comparison

# Collect metrics in a dictionary
metrics = {
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_logreg),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_gb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_logreg),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_logreg),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb)
    ]
}

# Create a DataFrame for clean comparison
comparison_df = pd.DataFrame(metrics)
comparison_df

Logistic Regression
Accuracy: 0.7927608232789212
Random Forest
Accuracy: 0.7771469127040455
Gradient Boosting
Accuracy: 0.7920511000709723


,Model,Accuracy,ROC-AUC,Precision,Recall,F1 Score
0,Logistic Regression,0.792761,0.841538,0.645390,0.486631,0.554878
1,Random Forest,0.777147,0.805153,0.593750,0.508021,0.547550
2,Gradient Boosting,0.792051,0.837257,0.631922,0.518717,0.569750
